# Example implementation of the model in Riley (1946)

Riley (1946) "Factors controlling phytoplankton populations on Georges Bank" is available [online](https://elischolar.library.yale.edu/journal_of_marine_research/624/).

The ODE presented in Riley (1946) is:

$$
\frac{dP}{dt} = P\left(P_h - R - G\right)
$$

Where:

* $P_h = p \, \frac{I_0}{k z_1} \, (1 - e^{-k z_1}) \, (1 - N) \, (1 - V)$: phytoplankton growth/increase
* $R = R_0 \, e^{r T}$: temperature-dependent respiration
* $G = g \, Z$: zooplankton grazing

Here, the model is just integrated forward. 3 slightly different model formulations are tested.

In [ ]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
measurements_riley1946 = pd.read_csv('data/Riley1946_measurements.csv')
curves_riley1946 = pd.read_csv('data/Riley1946_data-points-interpolated.csv')

In [ ]:
# ODE solution (RK4)
def riley1946_v1(P, it, state, parameters):
    # compute k * z1 using the state k
    kz1 = state['k'][it]*state['z1'][it]
    P_h = parameters['p']*state['I_0'][it]/kz1 * (1.0 - np.exp(-kz1)) * state['one_minus_N'][it] * min(1.0, state['z1'][it]/state['z2'][it])
    return P * (P_h - parameters['R0']*np.exp(parameters['r']*state['T'][it]) - parameters['g']*state['Z'][it])

def riley1946_v2(P, it, state, parameters):
    # compute k * z1 using the parameter k / Secchi disk reading
    k = parameters['k']/state['secchi'][it]
    kz1 = k*state['z1'][it]
    P_h = parameters['p']*state['I_0'][it]/kz1 * (1.0 - np.exp(-kz1)) * state['one_minus_N'][it] * min(1.0, state['z1'][it]/state['z2'][it])
    return P * (P_h - parameters['R0']*np.exp(parameters['r']*state['T'][it]) - parameters['g']*state['Z'][it])

def riley1946_v3(P, it, state, parameters):
    # test if "1-V" is indeed equivalent to min(1.0, z1/z2)
    k = parameters['k']/state['secchi'][it]
    kz1 = k*state['z1'][it]
    P_h = parameters['p']*state['I_0'][it]/kz1 * (1.0 - np.exp(-kz1)) * state['one_minus_N'][it] * state['one_minus_V'][it]
    return P * (P_h - parameters['R0']*np.exp(parameters['r']*state['T'][it]) - parameters['g']*state['Z'][it])

def rk4(f, x0, times, state, times_state, parameters):
    '''
    Perform the model integration using the Runge-Kutta 4.
    '''

    state_times = {v: np.interp(x=times, xp=times_state, fp=state[v]) for v in state}
    state_timeshalf = {v: np.interp(x=0.5*(times[1:]+times[:-1]), xp=times_state, fp=state[v]) for v in state}
    
    nt = len(times)
    x = np.zeros((nt, len(x0)))
    x[0, :] = x0
    dt = np.zeros(nt)
    dt[1:] = np.diff(times)
    for n in range(1, nt):
        k1 = f(x[n - 1, :], n-1, state_times, parameters) * dt[n]
        k2 = f(x[n - 1, :] + 0.5 * k1, n-1, state_timeshalf, parameters) * dt[n]
        k3 = f(x[n - 1, :] + 0.5 * k2, n-1, state_timeshalf, parameters) * dt[n]
        k4 = f(x[n - 1, :] + k3, n, state_times, parameters) * dt[n]
        x[n, :] = x[n - 1, :] + (k1 + 2 * k2 + 2 * k3 + k4) / 6

    return x

In [ ]:
parameters = {
    'p':   2.5,     # photosythetic constant (page 62)
    'R0':  0.0175,  # respiration rate at 0 °C (page 66)
    'g':   0.0075,  # rate of reduction of phytoplankton by a unit quantity of animals (page 67)
    'k':   1.7,     # extinction coefficient, to be divided by Secchi disk reading (page 62)
    'r':   0.069,   # rate of change of the respiratory rate with temperature (page 66)
}

dt = 1.0
t = np.arange(0, 366, dt)
P_v1 = rk4(riley1946_v1, np.array([3.4]), t, times_state=curves_riley1946['day'], state=curves_riley1946, parameters=parameters)
P_v2 = rk4(riley1946_v2, np.array([3.4]), t, times_state=curves_riley1946['day'], state=curves_riley1946, parameters=parameters)
P_v3 = rk4(riley1946_v3, np.array([3.4]), t, times_state=curves_riley1946['day'], state=curves_riley1946, parameters=parameters)

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

ax.plot(t, P_v1)
#ax.plot(t, P_v2)
ax.plot(t, P_v3)
ax.plot(measurements_riley1946['day'], measurements_riley1946['P'], ls='none', marker='o', color='black')
ax.set_ylim(bottom=0.0)
ax.set(xlabel='day of year', ylabel='phytoplankon (g C m$^{-2}$)', title='compare to Fig. 21 in Riley (1946)')
ax.grid(True)